# A quick evaluation on ESC, ESC*, and CTB

In [ ]:
import torch
import torch.nn as nn
from datasets import load_from_disk
from model import Causal_Model
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import compute_metrics

In [2]:
checkpoint = 'FacebookAI/roberta-large'

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
special_tokens_dict = {'additional_special_tokens': ['<e1>','</e1>','<e2>','</e2>']}
tokenizer.add_special_tokens(special_tokens_dict)

def tokenize_function_mask(examples):
    return tokenizer(examples["event_masked_sentence"], truncation=True)

def tokenize_function_tag(examples):
    return tokenizer(examples["event_tagged_sentence"], truncation=True)

## ESC evaluation demo

In [15]:
test_fold =load_from_disk('dataset/ESC_test_fold4')

masked_test_fold = test_fold.map(tokenize_function_mask, batched=True, batch_size=32)
masked_test_fold = masked_test_fold.remove_columns(['sentence', 'event_tagged_sentence', 'event_masked_sentence','e1','e2'])
masked_test_fold.set_format("torch")

tagged_test_fold = test_fold.map(tokenize_function_tag, batched=True, batch_size=32)
tagged_test_fold = tagged_test_fold.remove_columns(['sentence', 'event_tagged_sentence', 'event_masked_sentence','e1','e2'])
tagged_test_fold.set_format("torch")

print(f"test len: {len(masked_test_fold)}")

Map: 100%|██████████| 1399/1399 [00:00<00:00, 5547.24 examples/s]

test len: 1399


In [16]:
test_btz=20

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
dataloader_mask_test = DataLoader(
masked_test_fold, shuffle=False, batch_size=test_btz, collate_fn=data_collator)
dataloader_tag_test = DataLoader(
tagged_test_fold, shuffle=False,  batch_size=test_btz, collate_fn=data_collator)

dataloader_mask_test = tqdm(dataloader_mask_test, dynamic_ncols=True)

  0%|          | 0/70 [00:00<?, ?it/s]

In [19]:
device='cuda'
model=Causal_Model(bert_path=checkpoint, d_model=1024, num_heads=16, dropout_rate=0.5, device='cuda', visualize=False)

model.load_state_dict(torch.load('./checkpoints/ESC/best_model_fold4.pt'))
model=model.to(device)
criterion=nn.CrossEntropyLoss()

In [20]:
model.eval()
mean_loss_test = 0
predicted_all_test = []
gold_all_test = []
with torch.no_grad():
    iteration=0
    for mask_data, tag_data in zip(dataloader_mask_test, dataloader_tag_test):
        mask_data, tag_data=mask_data.to(device), tag_data.to(device)
        labels=tag_data['labels']
        labels=labels.to(device)

        del mask_data['labels']
        del tag_data['labels']
        
        outputs=model(mask_data, tag_data).squeeze(1)
        loss = criterion(outputs, labels)
        
        mean_loss_test = (mean_loss_test * iteration + loss.detach()) / (iteration + 1)
        iteration+=1

        predicted = torch.argmax(outputs, dim=-1)
        predicted=list(predicted.cpu().numpy())
        predicted_all_test+=predicted
        gold_all_test+=list(labels.cpu().numpy())
                                                    
precision_t, recall_t, f1_score_t = compute_metrics(gold_all_test, predicted_all_test)
print(f"[test ESC fold 4] p:{precision_t*100:.2f} r:{recall_t*100:.2f} F1:{f1_score_t*100:.2f} loss:{mean_loss_test.item():.4f}")

100%|██████████| 70/70 [02:01<00:00,  1.73s/it]  

[test ESC fold 4] p:64.62 r:77.40 F1:70.44 loss:1.5688


## ESC* evaluation demo

In [11]:
test_fold =load_from_disk('dataset/ESCstar_test_fold5')

masked_test_fold = test_fold.map(tokenize_function_mask, batched=True, batch_size=32)
masked_test_fold = masked_test_fold.remove_columns(['sentence', 'event_tagged_sentence', 'event_masked_sentence','e1','e2'])
masked_test_fold.set_format("torch")

tagged_test_fold = test_fold.map(tokenize_function_tag, batched=True, batch_size=32)
tagged_test_fold = tagged_test_fold.remove_columns(['sentence', 'event_tagged_sentence', 'event_masked_sentence','e1','e2'])
tagged_test_fold.set_format("torch")

print(f"test len: {len(masked_test_fold)}")

Map: 100%|██████████| 1399/1399 [00:00<00:00, 5178.28 examples/s]


Map: 100%|██████████| 1399/1399 [00:00<00:00, 5804.66 examples/s]

test len: 1399


In [12]:
test_btz=20

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
dataloader_mask_test = DataLoader(
masked_test_fold, shuffle=False, batch_size=test_btz, collate_fn=data_collator)
dataloader_tag_test = DataLoader(
tagged_test_fold, shuffle=False,  batch_size=test_btz, collate_fn=data_collator)

dataloader_mask_test = tqdm(dataloader_mask_test, dynamic_ncols=True)

  0%|          | 0/70 [00:00<?, ?it/s]

In [13]:
device='cuda'
model=Causal_Model(bert_path=checkpoint, d_model=1024, num_heads=16, dropout_rate=0.5, device='cuda', visualize=False)

model.load_state_dict(torch.load('./checkpoints/ESCstar/best_model_fold5.pt'))
model=model.to(device)
criterion=nn.CrossEntropyLoss()

In [14]:
model.eval()
mean_loss_test = 0
predicted_all_test = []
gold_all_test = []
with torch.no_grad():
    iteration=0
    for mask_data, tag_data in zip(dataloader_mask_test, dataloader_tag_test):
        mask_data, tag_data=mask_data.to(device), tag_data.to(device)
        labels=tag_data['labels']
        labels=labels.to(device)

        del mask_data['labels']
        del tag_data['labels']
        
        outputs=model(mask_data, tag_data).squeeze(1)
        loss = criterion(outputs, labels)
        
        mean_loss_test = (mean_loss_test * iteration + loss.detach()) / (iteration + 1)
        iteration+=1

        predicted = torch.argmax(outputs, dim=-1)
        predicted=list(predicted.cpu().numpy())
        predicted_all_test+=predicted
        gold_all_test+=list(labels.cpu().numpy())
                                                    
precision_t, recall_t, f1_score_t = compute_metrics(gold_all_test, predicted_all_test)
print(f"[test ESC* fold 5] p:{precision_t*100:.2f} r:{recall_t*100:.2f} F1:{f1_score_t*100:.2f} loss:{mean_loss_test.item():.4f}")

100%|██████████| 70/70 [02:01<00:00,  1.73s/it] 

[test ESC* fold 5] p:77.23 r:77.71 F1:77.47 loss:0.9138


## CTB evaluation demo

In [7]:
test_fold =load_from_disk('dataset/CTB_test_fold2')

masked_test_fold = test_fold.map(tokenize_function_mask, batched=True, batch_size=32)
masked_test_fold = masked_test_fold.remove_columns(['sentence', 'event_tagged_sentence', 'event_masked_sentence','e1','e2'])
masked_test_fold.set_format("torch")

tagged_test_fold = test_fold.map(tokenize_function_tag, batched=True, batch_size=32)
tagged_test_fold = tagged_test_fold.remove_columns(['sentence', 'event_tagged_sentence', 'event_masked_sentence','e1','e2'])
tagged_test_fold.set_format("torch")

print(f"test len: {len(masked_test_fold)}")

Map:   0%|          | 0/972 [00:00<?, ? examples/s]

Map: 100%|██████████| 972/972 [00:00<00:00, 5704.78 examples/s]

test len: 972


In [8]:
test_btz=20

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
dataloader_mask_test = DataLoader(
masked_test_fold, shuffle=False, batch_size=test_btz, collate_fn=data_collator)
dataloader_tag_test = DataLoader(
tagged_test_fold, shuffle=False,  batch_size=test_btz, collate_fn=data_collator)

dataloader_mask_test = tqdm(dataloader_mask_test, dynamic_ncols=True)

  0%|          | 0/49 [00:00<?, ?it/s]

In [5]:
device='cuda'
model=Causal_Model(bert_path=checkpoint, d_model=1024, num_heads=16, dropout_rate=0.5, device='cuda', visualize=False)

model.load_state_dict(torch.load('./checkpoints/CTB/best_model_fold2.pt'))
model=model.to(device)
criterion=nn.CrossEntropyLoss()

In [9]:
model.eval()
mean_loss_test = 0
predicted_all_test = []
gold_all_test = []
with torch.no_grad():
    iteration=0
    for mask_data, tag_data in zip(dataloader_mask_test, dataloader_tag_test):
        mask_data, tag_data=mask_data.to(device), tag_data.to(device)
        labels=tag_data['labels']
        labels=labels.to(device)

        del mask_data['labels']
        del tag_data['labels']
        
        outputs=model(mask_data, tag_data).squeeze(1)
        loss = criterion(outputs, labels)
        
        mean_loss_test = (mean_loss_test * iteration + loss.detach()) / (iteration + 1)
        iteration+=1

        predicted = torch.argmax(outputs, dim=-1)
        predicted=list(predicted.cpu().numpy())
        predicted_all_test+=predicted
        gold_all_test+=list(labels.cpu().numpy())
                                                    
precision_t, recall_t, f1_score_t = compute_metrics(gold_all_test, predicted_all_test)
print(f"[test CTB fold 2] p:{precision_t*100:.2f} r:{recall_t*100:.2f} F1:{f1_score_t*100:.2f} loss:{mean_loss_test.item():.4f}")

100%|██████████| 49/49 [00:20<00:00,  2.44it/s]

[test CTB fold 2] p:60.71 r:89.47 F1:72.34 loss:0.1075
